In [1]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler

df = pd.read_csv("../data/diabetes_prediction_dataset.csv")
print("Original Shape:", df.shape)
df.head()

Original Shape: (100000, 9)


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,Female,80.0,0,1,never,25.19,6.6,140,0
1,Female,54.0,0,0,No Info,27.32,6.6,80,0
2,Male,28.0,0,0,never,27.32,5.7,158,0
3,Female,36.0,0,0,current,23.45,5.0,155,0
4,Male,76.0,1,1,current,20.14,4.8,155,0


In [ ]:
#  DUPLICATES
print("Duplicate rows:", df.duplicated().sum())
df = df.drop_duplicates()
print("After removing duplicates:", df.shape)

Duplicate rows: 3854
After removing duplicates: (96146, 9)


In [3]:
# ENCODING

# Gender — Label Encoding
print("Gender unique:", df['gender'].unique())
df['gender'] = df['gender'].map({'Male': 1, 'Female': 0, 'Other': 2})

# Smoking History — Ordinal Encoding (risk order)
smoking_order = {
    'No Info'     : 0,
    'never'       : 1,
    'former'      : 2,
    'ever'        : 3,
    'not current' : 4,
    'current'     : 5
}
df['smoking_history'] = df['smoking_history'].map(smoking_order)

print("\nEncoding done!")
print(df.dtypes)
df.head()

Gender unique: <StringArray>
['Female', 'Male', 'Other']
Length: 3, dtype: str

Encoding done!
gender                   int64
age                    float64
hypertension             int64
heart_disease            int64
smoking_history          int64
bmi                    float64
HbA1c_level            float64
blood_glucose_level      int64
diabetes                 int64
dtype: object


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,0,80.0,0,1,1,25.19,6.6,140,0
1,0,54.0,0,0,0,27.32,6.6,80,0
2,1,28.0,0,0,1,27.32,5.7,158,0
3,0,36.0,0,0,5,23.45,5.0,155,0
4,1,76.0,1,1,5,20.14,4.8,155,0


In [4]:
# OUTLIER TREATMENT

# BMI > 60 — extreme outliers
print("BMI > 60:", len(df[df['bmi'] > 60]))

# IQR Method for BMI
Q1 = df['bmi'].quantile(0.25)
Q3 = df['bmi'].quantile(0.75)
IQR = Q3 - Q1
upper_bound = Q3 + 3 * IQR  # 3*IQR — conservative

print(f"Q1: {Q1}, Q3: {Q3}, IQR: {IQR}")
print(f"Upper Bound: {upper_bound}")
print(f"Rows above upper bound: {len(df[df['bmi'] > upper_bound])}")

# Cap karo — drop nahi karo
df['bmi'] = np.where(df['bmi'] > upper_bound, upper_bound, df['bmi'])
print("\nBMI after capping:")
print(df['bmi'].describe())

BMI > 60: 115
Q1: 23.4, Q3: 29.86, IQR: 6.460000000000001
Upper Bound: 49.24
Rows above upper bound: 905

BMI after capping:
count    96146.000000
mean        27.269889
std          6.551379
min         10.010000
25%         23.400000
50%         27.320000
75%         29.860000
max         49.240000
Name: bmi, dtype: float64


In [5]:
# SCALING

columns_to_scale = [
    'age',
    'bmi',
    'HbA1c_level',
    'blood_glucose_level'
]

# Save original values for reference
df_original = df.copy()

scaler = StandardScaler()
df[columns_to_scale] = scaler.fit_transform(df[columns_to_scale])

print("Scaling done!")
print(df[columns_to_scale].describe().round(2))

Scaling done!
            age       bmi  HbA1c_level  blood_glucose_level
count  96146.00  96146.00     96146.00             96146.00
mean       0.00     -0.00        -0.00                -0.00
std        1.00      1.00         1.00                 1.00
min       -1.86     -2.63        -1.89                -1.42
25%       -0.79     -0.59        -0.68                -0.93
50%        0.05      0.01         0.25                 0.04
75%        0.77      0.40         0.62                 0.51
max        1.70      3.35         3.23                 3.95


In [7]:
# SAVE CLEANED DATA

df.to_csv("../data/diabetes_cleaned.csv", index=False)
df_original.to_csv("../data/diabetes_original_encoded.csv", index=False)

print("Files saved!")
print("diabetes_cleaned.csv       → Scaled data (for ML)")
print("diabetes_original_encoded.csv → Unscaled (for Clustering/Association)")
print("\nFinal Shape:", df.shape)
df.head()

Files saved!
diabetes_cleaned.csv       → Scaled data (for ML)
diabetes_original_encoded.csv → Unscaled (for Clustering/Association)

Final Shape: (96146, 9)


,gender,age,hypertension,heart_disease,smoking_history,bmi,HbA1c_level,blood_glucose_level,diabetes
0,0,1.700840,0,1,1,-0.317475,0.994563,0.043554,0
1,0,0.543372,0,0,0,0.007649,0.994563,-1.423096,0
2,1,-0.614096,0,0,1,0.007649,0.155970,0.483549,0
3,0,-0.257952,0,0,5,-0.583069,-0.496269,0.410216,0
4,1,1.522768,1,1,5,-1.088309,-0.682623,0.410216,0


In [9]:
import joblib

# Scaler save karo
joblib.dump(scaler, '../models/scaler.pkl')
print("Scaler saved")

Scaler saved


In [ ]:
# diabetes_cleaned.csv  -> Classification + Regression
# diabetes_original_encoded.csv -> Clustering + Association Rules